# Plotting Time Series Models

In [54]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob

In [55]:
# set folder with data
path = '../results/'
# get all files in the folder
all_files = glob.glob(path + "*.csv")

In [56]:
# filter the files that contain "covid"
covid_files = [f for f in all_files if 'covid' in f]
# filter the files that contain "epidemics"
epidemics_files = [f for f in all_files if 'epidemics' in f]

In [60]:
# get the original data
df_covid = pd.read_csv('../Data/silver/covid_data_weekly.csv')  

In [61]:
nrow_dict = {}
for country in df_covid['country'].unique():
    df_temp = df_covid[df_covid['country'] == country]
    nrow_dict[country] = len(df_temp)

In [62]:
nrow_dict

{'Brazil': 164,
 'Chile': 164,
 'Colombia': 164,
 'Dominican Republic': 164,
 'Germany': 164,
 'Italy': 164,
 'Mexico': 164,
 'Panama': 164,
 'US': 164,
 'Uruguay': 164}

In [22]:
from functools import reduce

# -------------------------------------------------------------------
# 1.  Load raw CSVs
# -------------------------------------------------------------------
df_dengue    = pd.read_csv('../Data/silver/dengue_no_split.csv')
df_zika      = pd.read_csv('../Data/silver/zika.csv').drop(columns=['Unnamed: 0'])
df_chic      = pd.read_csv('../Data/silver/chicunguya.csv').drop(columns=['Unnamed: 0'])
df_varicela  = pd.read_csv('../Data/silver/varicela.csv')

# -------------------------------------------------------------------
# 2.  Helper that:  (i) parses DATE  (ii) builds cumulative + t-1 series
# -------------------------------------------------------------------
def prep(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    df = df.copy()
    df['DATE'] = pd.to_datetime(df['DATE'])
    df = df.sort_values('DATE')

    # cumulative sum of new weekly cases
    df[f'cum_{prefix}'] = df['Casos'].cumsum()

    # next-week value (shifted *backwards* so that current row “knows” next week)
    df[f'cum_{prefix}_t_1'] = df[f'cum_{prefix}'].shift(-1)

    # keep only the useful columns
    return df[['DATE', f'cum_{prefix}', f'cum_{prefix}_t_1']]

# -------------------------------------------------------------------
# 3.  Prepare each disease frame
# -------------------------------------------------------------------
dengue   = prep(df_dengue,   'dengue')
zika     = prep(df_zika,     'zika')
chik     = prep(df_chic,     'chik')
varicela = prep(df_varicela, 'var')

# -------------------------------------------------------------------
# 4.  Outer-merge on DATE → aligns differing timelines, fills gaps with NaN
# -------------------------------------------------------------------
df_epidemics = reduce(
    lambda left, right: pd.merge(left, right, on='DATE', how='outer'),
    [dengue, zika, chik, varicela]
).sort_values('DATE').set_index('DATE')

# -------------------------------------------------------------------
# 5.  Final check
# -------------------------------------------------------------------
print(df_epidemics.head())
print(df_epidemics.tail())


            cum_dengue  cum_dengue_t_1  cum_zika  cum_zika_t_1  cum_chik  \
DATE                                                                       
2007-01-01         315           551.0       NaN           NaN       NaN   
2007-01-08         551           803.0       NaN           NaN       NaN   
2007-01-15         803          1108.0       NaN           NaN       NaN   
2007-01-22        1108          1468.0       NaN           NaN       NaN   
2007-01-29        1468          1831.0       NaN           NaN       NaN   

            cum_chik_t_1  cum_var  cum_var_t_1  
DATE                                            
2007-01-01           NaN    250.0        557.0  
2007-01-08           NaN    557.0        858.0  
2007-01-15           NaN    858.0       1124.0  
2007-01-22           NaN   1124.0       1481.0  
2007-01-29           NaN   1481.0       1763.0  
            cum_dengue  cum_dengue_t_1  cum_zika  cum_zika_t_1  cum_chik  \
DATE                                            

In [23]:
df_covid[df_covid['country'] == 'Italy'].tail(5)

,country,date,new_cases,cases
979,Italy,2023-02-06,30901,25519067
980,Italy,2023-02-13,28347,25547414
981,Italy,2023-02-20,29438,25576852
982,Italy,2023-02-27,26658,25603510
983,Italy,2023-03-06,0,25603510


# Wrangle

## Covid

In [24]:
df_covid.tail()

,country,date,new_cases,cases
1635,Uruguay,2023-02-06,555,1033265
1636,Uruguay,2023-02-13,501,1033766
1637,Uruguay,2023-02-20,280,1034046
1638,Uruguay,2023-02-27,257,1034303
1639,Uruguay,2023-03-06,0,1034303


In [25]:
df_covid.head()

,country,date,new_cases,cases
0,Brazil,2020-01-20,0,0
1,Brazil,2020-01-27,0,0
2,Brazil,2020-02-03,0,0
3,Brazil,2020-02-10,0,0
4,Brazil,2020-02-17,0,0


In [26]:
# create cases in t+1 for each country
df_covid['cases_t_1'] = df_covid.groupby('country')['cases'].shift(-1)

In [27]:
# drop nan based in cases_t_1
df_covid = df_covid.dropna(subset=['cases_t_1'])

In [28]:
dict_n_rows = {}
# count observations by country
for country in df_covid['country']:
    df_temp = df_covid[df_covid['country'] == country]
    dict_n_rows[country] = len(df_temp)

dict_n_rows

{'Brazil': 163,
 'Chile': 163,
 'Colombia': 163,
 'Dominican Republic': 163,
 'Germany': 163,
 'Italy': 163,
 'Mexico': 163,
 'Panama': 163,
 'US': 163,
 'Uruguay': 163}

In [29]:
# ----------------------------------------------------------
# 1.  Make sure date is datetime & records are chronological
# ----------------------------------------------------------
df_covid['date'] = pd.to_datetime(df_covid['date'])
df_covid = df_covid.sort_values('date')

# ----------------------------------------------------------
# 2.  Pivot: rows = date, columns = country, values = cases_t_1
#     (all countries have the same number of observations)
# ----------------------------------------------------------
wide = (
    df_covid
      .pivot(index='date', columns='country', values='cases_t_1')
      .sort_index()                       # chronological index
)

# ----------------------------------------------------------
# 3.  Mark the first 80 % rows as TRAIN and the last 20 % as TEST
# ----------------------------------------------------------
split_point = int(len(wide) * 0.8)        # row where the 80/20 cut happens

wide['set'] = np.where(
    np.arange(len(wide)) < split_point,   # row-position test
    'train',
    'test'
)

# ----------------------------------------------------------
# 4.  (Optional) quick sanity check
# ----------------------------------------------------------
print(wide['set'].value_counts(normalize=True))
print(wide.head())


set
train    0.797546
test     0.202454
Name: proportion, dtype: float64
country     Brazil  Chile  Colombia  Dominican Republic  Germany   Italy  \
date                                                                       
2020-01-20     0.0    0.0       0.0                 0.0     10.0     2.0   
2020-01-27     0.0    0.0       0.0                 0.0     14.0     3.0   
2020-02-03     0.0    0.0       0.0                 0.0     16.0     3.0   
2020-02-10     0.0    2.0       0.0                 0.0     16.0   155.0   
2020-02-17     2.0    9.0       0.0                 1.0    117.0  1694.0   

country     Mexico  Panama    US  Uruguay    set  
date                                              
2020-01-20     0.0     0.0   7.0      0.0  train  
2020-01-27     0.0     0.0  11.0      0.0  train  
2020-02-03     0.0     0.0  13.0      0.0  train  
2020-02-10     0.0     0.0  15.0      0.0  train  
2020-02-17     5.0     0.0  31.0      0.0  train  


In [30]:
wide.tail()

country,Brazil,Chile,Colombia,Dominican Republic,Germany,Italy,Mexico,Panama,US,Uruguay,set
date,,,,,,,,,,,
2023-01-30,36932830.0,5138732.0,6355637.0,660412.0,37907312.0,25519067.0,7400848.0,1030214.0,102862878.0,1033265.0,test
2023-02-06,36987682.0,5149301.0,6356468.0,660492.0,38002114.0,25547414.0,7429778.0,1030658.0,103136076.0,1033766.0,test
2023-02-13,37020531.0,5163316.0,6357173.0,660533.0,38111063.0,25576852.0,7450992.0,1031014.0,103382762.0,1034046.0,test
2023-02-20,37081209.0,5180329.0,6358232.0,660705.0,38210851.0,25603510.0,7470653.0,1031273.0,103646974.0,1034303.0,test
2023-02-27,37076053.0,5192286.0,6359093.0,660790.0,38249060.0,25603510.0,7483444.0,1031731.0,103802701.0,1034303.0,test


In [31]:
df_covid_w = wide.copy().reset_index()

## Epidemics

In [32]:
idx = df_epidemics.index
n_rows     = len(idx)
split_pt   = int(n_rows * 0.8)          # 80 % train, 20 % test
train_idx  = idx[:split_pt]             # older 80 %
test_idx   = idx[split_pt:]             # most-recent 20 %

df_epidemics.loc[train_idx, 'set'] = 'train'
df_epidemics.loc[test_idx,  'set'] = 'test'

# Plots

## Covid

In [33]:
df_covid_w.head()

country,date,Brazil,Chile,Colombia,Dominican Republic,Germany,Italy,Mexico,Panama,US,Uruguay,set
0,2020-01-20,0.0,0.0,0.0,0.0,10.0,2.0,0.0,0.0,7.0,0.0,train
1,2020-01-27,0.0,0.0,0.0,0.0,14.0,3.0,0.0,0.0,11.0,0.0,train
2,2020-02-03,0.0,0.0,0.0,0.0,16.0,3.0,0.0,0.0,13.0,0.0,train
3,2020-02-10,0.0,2.0,0.0,0.0,16.0,155.0,0.0,0.0,15.0,0.0,train
4,2020-02-17,2.0,9.0,0.0,1.0,117.0,1694.0,5.0,0.0,31.0,0.0,train


In [37]:
len(df_covid_w)

163

In [67]:
# First, make sure 'date' is actually parsed as datetime (recommended)
df_test['date'] = pd.to_datetime(df_test['date'])

# Then check for duplicates in the 'date' column
duplicates = df_test[df_test['date'].duplicated()]

# Print how many there are and inspect them
print(f"Number of duplicated dates: {len(duplicates)}")
print(duplicates.head())


Number of duplicated dates: 0
Empty DataFrame
Columns: [date, Brazil, Chile, Colombia, Dominican Republic, Germany, Italy, Mexico, Panama, US, Uruguay]
Index: []


In [64]:
len(df_test)

164

In [ ]:
# see if df_test has duplicates


AttributeError: 'DataFrame' object has no attribute 'duplicates'

In [39]:
# change column names from 1: df_test, to all lower cap
df_test.columns = [col.lower() for col in df_test.columns]

In [42]:
df_test.columns

Index(['date', 'brazil', 'chile', 'colombia', 'dominican republic', 'germany',
       'italy', 'mexico', 'panama', 'us', 'uruguay'],
      dtype='object')

In [45]:
df_test_br = df_test[['date','brazil']].copy()
df_covid_br = df_covid_w[['date','Brazil']].copy()

In [50]:
df_test_br['date'] = pd.to_datetime(df_test_br['date'])

In [51]:
# merge df_covid_w and df_test by date
df_merge_test = df_covid_br.merge(df_test_br, how = 'outer', on = 'date')

In [53]:
df_merge_test


,date,Brazil,brazil
0,2020-01-20,0.0,0.000000e+00
1,2020-01-27,0.0,0.000000e+00
2,2020-02-03,0.0,0.000000e+00
3,2020-02-10,0.0,0.000000e+00
4,2020-02-17,2.0,0.000000e+00
...,...,...,...
159,2023-02-06,36987682.0,4.122488e+07
160,2023-02-13,37020531.0,4.150247e+07
161,2023-02-20,37081209.0,4.178012e+07
162,2023-02-27,37076053.0,4.205782e+07


In [36]:
len(df_test)

164

In [ ]:
df_test.tail()

In [ ]:
df_covid_w.tail()

In [ ]:
print(len(df_test))
print(len(df_covid_w))